# 04b - Model Serving & Deployment: Service Orders (Tabular)

**Azure ML MLOps Workshop | Track B - Tabular Data**

### Context

In Notebook 04, you deployed a text classifier that accepts inspection comments.
Now you'll deploy a **tabular classifier** that accepts structured service order
fields and predicts the repair type.

### What you'll do:
1. Create a second managed online endpoint (`contoso-repair-classifier`)
2. Deploy the tabular model with its own scoring script
3. Test with structured JSON payloads

### Key difference from Track A:
- **Track A** scoring script: cleans text, applies TF-IDF vectorizer, predicts
- **Track B** scoring script: receives structured fields, applies sklearn preprocessor, predicts
- **Same deployment infrastructure:** endpoint, deployment, traffic routing — all identical

---

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import json

SUBSCRIPTION_ID = "<YOUR_SUBSCRIPTION_ID>"  # <<<< CHANGE THIS TO YOUR AZURE SUBSCRIPTION ID
RESOURCE_GROUP = "<YOUR_RESOURCE_GROUP>"  # <<<< CHANGE THIS TO YOUR RESOURCE GROUP (e.g., rg-aml-workshop-jd)
WORKSPACE_NAME = "<YOUR_WORKSPACE_NAME>"  # <<<< CHANGE THIS TO YOUR WORKSPACE NAME (e.g., aml-workshop-jd)

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)
print(f"Connected to: {ml_client.workspace_name}")

## Step 1: Create the Online Endpoint

A **separate endpoint** from the text classifier — in production, different
models serve different business functions via different REST URLs.

In [ ]:
from azure.ai.ml.entities import ManagedOnlineEndpoint

ENDPOINT_NAME = "contoso-repair-classifier"  # <<<< CHANGE THIS - ADD YOUR INITIALS (e.g., contoso-repair-classifier-jd)

endpoint = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="Contoso service orders repair type classification endpoint",
    auth_mode="key",
    tags={
        "region": "region-1",
        "use_case": "repair-type-prediction",
        "team": "contoso-data-science",
        "data_type": "tabular",
    },
)

endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"Endpoint created: {endpoint.name}")
print(f"  Scoring URI: {endpoint.scoring_uri}")
print(f"  State: {endpoint.provisioning_state}")

## Step 2: Create a Deployment (Blue)

In [ ]:
from azure.ai.ml.entities import (
    ManagedOnlineDeployment,
    CodeConfiguration,
    Environment,
)

MODEL_NAME = "contoso-repair-classifier"

blue_deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=f"azureml:{MODEL_NAME}@latest",
    code_configuration=CodeConfiguration(
        code="../../src/track_b_tabular",
        scoring_script="score_os.py",
    ),
    environment=Environment(
        conda_file="../../environment/track_b_tabular/deployment_env_os.yml",
        image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest",
    ),
    instance_type="Standard_DS2_v2",
    instance_count=1,
)

print("Creating deployment (this takes 5-10 minutes)...")
blue_deployment = ml_client.online_deployments.begin_create_or_update(blue_deployment).result()
print(f"Deployment created: {blue_deployment.name}")

In [ ]:
# Route 100% traffic to the blue deployment
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Traffic routing: 100% -> blue")

## Step 3: Test the Endpoint

Send structured service order records — compare this payload format
to the text-based payload in Track A.

In [ ]:
import tempfile, os

test_orders = [
    {"EquipmentModel": "EX200", "JobCode": "PM", "ServiceCenter": "1001", "QtyOrdered": 4.0, "month": 12, "quarter": 4, "day_of_week": 0},
    {"EquipmentModel": "DZ300", "JobCode": "RP", "ServiceCenter": "1001", "QtyOrdered": 1.0, "month": 6, "quarter": 2, "day_of_week": 3},
    {"EquipmentModel": "LH400", "JobCode": "CM", "ServiceCenter": "1001", "QtyOrdered": 12.0, "month": 3, "quarter": 1, "day_of_week": 1},
    {"EquipmentModel": "TR500", "JobCode": "IN", "ServiceCenter": "1001", "QtyOrdered": 2.0, "month": 9, "quarter": 3, "day_of_week": 4},
    {"EquipmentModel": "EX100", "JobCode": "PM", "ServiceCenter": "1001", "QtyOrdered": 8.0, "month": 1, "quarter": 1, "day_of_week": 2},
]

request_payload = json.dumps({"data": test_orders})

with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
    f.write(request_payload)
    request_file_path = f.name

response = ml_client.online_endpoints.invoke(
    endpoint_name=ENDPOINT_NAME,
    deployment_name="blue",
    request_file=request_file_path,
)

os.remove(request_file_path)

results = json.loads(response)
print("Predictions:")
print("-" * 80)
for r in results["results"]:
    repair = r["predicted_repair_type"]
    conf = r["confidence"]
    prob = r["overhaul_probability"]
    modelo = r["input"]["EquipmentModel"]
    print(f"  {modelo:8s} | {repair:20s} | prob_overhaul: {prob:.2f} | confidence: {conf}")

## Step 4: Compare Both Endpoints Side by Side

In [ ]:
print("=== Active Endpoints ===")
print()
for ep in ml_client.online_endpoints.list():
    print(f"Endpoint: {ep.name}")
    print(f"  Use case:    {ep.tags.get('use_case', 'N/A')}")
    print(f"  Data type:   {ep.tags.get('data_type', 'text')}")
    print(f"  Scoring URI: {ep.scoring_uri}")
    print()

## Step 5: Check Endpoint Logs

In [ ]:
logs = ml_client.online_deployments.get_logs(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    lines=50,
)
print("Recent deployment logs:")
print(logs)

## Go to Azure ML Studio Now

Navigate to **Endpoints** and you should see **two endpoints**:

| Endpoint | Input Type | Model |
|----------|-----------|-------|
| `contoso-lead-classifier` | text comments | Text classifier (TF-IDF) |
| `contoso-repair-classifier` | Structured JSON fields | Tabular classifier (OneHotEncoder) |

The deployment infrastructure is identical — the only difference is the scoring script.

---

## Key Takeaways

| Concept | What We Did |
|---------|------------|
| **Multiple endpoints** | Each model gets its own stable REST URL |
| **Scoring script** | `score.py` (text) vs `score_os.py` (tabular) — different preprocessing, same framework |
| **Same infrastructure** | Managed endpoints, blue/green, autoscaling — data-type agnostic |
| **Integration** | Both endpoints return JSON — any downstream app can consume them |

---

**Next:** [05b - Model Monitoring (Tabular)](./05b_model_monitoring_tabular.ipynb)

## Cleanup (run after workshop)

Endpoints incur cost while running. Delete when done.

In [ ]:
# Uncomment to delete the endpoint after the workshop
# ml_client.online_endpoints.begin_delete(name=ENDPOINT_NAME).result()
# print(f"Endpoint '{ENDPOINT_NAME}' deleted.")